# Understanding the Multi-Layer Perceptron (MLP)

This notebook provides a detailed explanation of the math and mechanics behind each function in our `MLP` class.

## 1. Activation Functions
Before diving into the MLP methods, let's understand the activation functions used to introduce non-linearity:

### 1.1 ReLU (Rectified Linear Unit)
**Formula:** $f(x) = \max(0, x)$

**Explanation:** ReLU simply outputs the input directly if it is positive, otherwise, it outputs zero. This helps the network learn complex patterns while avoiding the vanishing gradient problem common with smooth functions like Sigmoid or Tanh.

### 1.2 Softmax
**Formula:** $\sigma(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$

**Explanation:** Softmax is used in the output layer for multi-class classification. It converts a vector of raw scores (logits) into a probability distribution where all values are between 0 and 1, and their sum equals 1.

In [ ]:
import numpy as np

def relu(z):
    return np.maximum(0, z)

def softmax(z):
    z_shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

## 2. Parameter Initialization (`initialize_parameters`)

Neural networks are sensitive to how weights are initialized. Here are some common techniques:
- **Zero Initialization:** Setting all weights to 0. *Problem:* All neurons learn the same features during training (symmetry problem).
- **Random Initialization:** Small random values. *Problem:* Can lead to vanishing or exploding gradients in deep networks.
- **Xavier/Glorot Initialization:** Variance is scaled by $1/N_{in}$. Good for Sigmoid/Tanh activations.
- **He Initialization:** Designed specifically to perform well with ReLU activations.

### Mathematical Formulation of He Initialization
For a specific layer with $N_{in}$ input units (fan-in), the weights are drawn from a normal distribution with:
- **Mean** $\mu = 0$
- **Variance** $\sigma^2 = \frac{2}{N_{in}}$
So, $W \sim \mathcal{N}\left(0, \sqrt{\frac{2}{N_{in}}}\right)$

**Why?** ReLU sets half of the activations to zero. To keep the variance of the forward pass consistently distributed across layers, He initialization multiplies the variance by 2 compared to Xavier.

In [ ]:
def initialize_parameters(self):
    '''Example layout of He Initialization for weights and zeros for biases.'''
    np.random.seed(42)
    
    # Layer 1 weights: shape (input_size, hidden1_size)
    # np.random.randn gives standard normal distribution.
    # Multiplying by sqrt(2/fan_in) gives us He Initialization.
    self.W1 = np.random.randn(self.sizes[0], self.sizes[1]) * np.sqrt(2. / self.sizes[0])
    self.b1 = np.zeros((1, self.sizes[1]))  # Biases are typically initialized to 0

## 3. Forward Propagation (`forward_propagation`)

During the forward pass, the model makes predictions by passing data through its layers.

### Storing Weight and Bias Matrices
For a transition from a layer with $N_{in}$ neurons to $N_{out}$ neurons:
- **Weight Matrix ($W$):** Shape is `(N_in, N_out)`. Each column represents the weights connecting all inputs to a single neuron in the next layer.
- **Bias Vector ($b$):** Shape is `(1, N_out)`. A single bias term is added to each neuron in the layer.

### How the Dot Product Works
The linear transformation is: $Z = X \cdot W + b$

**Example:**
Let's say we have $1$ sample with $2$ features, feeding into a layer with $3$ neurons.

Let $X = [1, 2]$ (Shape: `1x2`)

Let $W = \begin{bmatrix} 0.1 & 0.2 & 0.3 \\ 0.4 & 0.5 & 0.6 \end{bmatrix}$ (Shape: `2x3`)

Let $b = [0.1, 0.1, 0.1]$ (Shape: `1x3`)

First, compute the Dot Product $X \cdot W$:  
$[1(0.1)+2(0.4),\; 1(0.2)+2(0.5),\; 1(0.3)+2(0.6)] = [0.9, 1.2, 1.5]$

Next, add the bias $b$:  
$Z = [0.9+0.1, 1.2+0.1, 1.5+0.1] = [1.0, 1.3, 1.6]$

Finally, the activation function (like ReLU) is applied element-wise: $A = \text{ReLU}(Z) = [1.0, 1.3, 1.6]$.

## 4. Compute Loss (`compute_loss`)

Loss functions measure how far our network's predictions are from actual labels.

### Different Types of Loss Functions
1. **Mean Squared Error (MSE):** Averages the squared differences. Used primarily for Regression tasks.
2. **Mean Absolute Error (MAE):** Averages the absolute differences. Used for Regression (more robust to outliers than MSE).
3. **Binary Cross-Entropy (BCE):** Used for binary classification.
4. **Categorical Cross-Entropy (CCE):** Used for multi-class classification.

### Categorical Cross-Entropy Mechanics & Math
**Formula:** $L = -\frac{1}{m} \sum_{i=1}^{m} \sum_{k=1}^{K} y_{i,k} \log(\hat{y}_{i,k})$
Where:
- $m$: number of samples in the batch
- $K$: number of classes
- $y_{i,k}$: 1 if the true class of sample $i$ is $k$, else 0 (derived from one-hot encoding)
- $\hat{y}_{i,k}$: predicted probability that sample $i$ belongs to class $k$ (output of Softmax)

**Explanation:**
Because $y_{i,k}$ is $1$ only for the true class and $0$ for all others, the inner sum simply evaluates to $\log(\hat{y}_{true})$. 

- If the prediction is confident and correct ($\hat{y}_{true} \approx 1$), taking the negative log is near $0$ (low loss).
- If the prediction is completely wrong ($\hat{y}_{true} \approx 0$), the negative log approaches $\infty$, yielding a very high penalty (loss). 

The function sums this up across all $m$ samples and takes the average.

In [ ]:
def compute_loss(self, y_true_one_hot, y_pred):
    '''Categorical Cross-Entropy Loss computation'''
    m = y_true_one_hot.shape[0]
    
    # To avoid taking log(0) which is undefined/infinity, we add a tiny epsilon
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1. - epsilon)
    
    # Math: -1/m * sum(y_true * log(y_pred))
    loss = -1/m * np.sum(y_true_one_hot * np.log(y_pred))
    return loss

## 5. Backward Propagation (`backward_propagation`)

Backpropagation computes the gradient of the loss function with respect to the weights by applying the **Chain Rule** from calculus backwards through the network.

### Mathematical Steps for a Hidden Layer:
1. **Upstream Gradient ($dA$):** We receive $dA$, which tells us how much the Loss changes relative to our layer's activation output.
2. **Local Gradient (Activation):** $dZ = dA * \text{derivative}(Z)$. For ReLU, the derivative is $1$ if $Z > 0$ else $0$. This acts like a switch passing the gradient backwards only if the neuron was active.
3. **Gradients w.r.t Parameters:**
   - $dW = \frac{1}{m} (A_{prev}^T \cdot dZ)$
   - $db = \frac{1}{m} \sum dZ$
4. **Gradient for Next Layer Down:** $dA_{prev} = dZ \cdot W^T$

*(Special case: When computing the derivative of Categorical Cross-Entropy with respect to the inputs to the Softmax layer, the complex chain rule math perfectly and elegantly simplifies to just $dZ_{output} = A_{output} - Y_{true}$).* 


In [ ]:
def backward_propagation(self, X, y_true_one_hot):
    m = X.shape[0]
    
    # 1. Output Layer Gradient (Fast simplified Softmax + CCE derivative)
    dZ3 = self.A3 - y_true_one_hot
    self.dW3 = (1/m) * np.dot(self.A2.T, dZ3)
    self.db3 = (1/m) * np.sum(dZ3, axis=0, keepdims=True)
    
    # 2. Hidden Layer Gradients (Chain rule with ReLU)
    dA2 = np.dot(dZ3, self.W3.T)
    dZ2 = dA2 * (self.Z2 > 0).astype(float) # ReLU Derivative
    self.dW2 = (1/m) * np.dot(self.A1.T, dZ2)
    self.db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)

## 6. Update Parameters (`update_parameters`)

After calculating $dW$ and $db$ representing the 'slopes' for all layers, we update the actual weights and biases using **Gradient Descent**.

### Math Formulation:
$W_{new} = W_{old} - \alpha \cdot dW$
$b_{new} = b_{old} - \alpha \cdot db$

Where **$\alpha$ is the Learning Rate** (a hyperparameter that controls the step size).

**Why subtract?** 
The gradient points in the direction of the steepest *ascent* (the direction where loss increases most rapidly). Because our goal in training is to minimize the Loss, we must step in the **opposite** direction of the gradient.

In [ ]:
def update_parameters(self, lr):
    '''Gradient Descent step'''
    self.W1 -= lr * self.dW1
    self.b1 -= lr * self.db1
    
    self.W2 -= lr * self.dW2
    self.b2 -= lr * self.db2
    
    self.W3 -= lr * self.dW3
    self.b3 -= lr * self.db3